In [ ]:
import ee

In [ ]:
ee.Authenticate()

In [ ]:
ee.Initialize(project="YOUR_GCP_PROJECT")

In [ ]:
ic = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")

In [ ]:
def export_alpha_patch(
    year: int,
    center_lon: float,
    center_lat: float,
    size_m: float = 10_000, 
    scale: float = 10,
    crs: str = "EPSG:6933",      
    folder: str = "alphaearth",
    description_prefix: str = "alphaearth_patch"
):
    aoi = ee.Geometry.Point([center_lon, center_lat]).buffer(size_m / 2).bounds()
    print(aoi.getInfo())

    ic_year = (ic
        .filterDate(f"{year}-01-01", f"{year+1}-01-01")
        .filterBounds(aoi)
    )
    ae_img = ic_year.mosaic()  # 64

    print("bands:", ae_img.bandNames().size().getInfo())

    ae_clip = ae_img.clip(aoi)
    print(ae_clip.getInfo())

    task = ee.batch.Export.image.toDrive(
        image=ae_clip,
        description=f"{description_prefix}_{year}_{int(size_m/1000)}km_2",
        folder=folder,
        fileNamePrefix=f"{description_prefix}_{year}_{int(size_m/1000)}km_2",
        region=aoi,              # IMPORTANT: export only this square
        scale=scale,             # 10m
        crs=crs,                 # set output CRS (use EPSG:4326 if you prefer lat/lon)
        maxPixels=1e13
        # You can also add: fileFormat="GeoTIFF"  (default is GeoTIFF)
    )
    task.start()
    return task

In [ ]:
proj = ee.Projection('PROJCS["WGS 84 / NSIDC EASE-Grid 2.0 Global",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Cylindrical_Equal_Area"],PARAMETER["standard_parallel_1",30],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","6933"]]')

In [ ]:
display(proj)

In [ ]:
task = export_alpha_patch(
    year=2024,
    center_lon=-85.424818,
    center_lat=42.630143, 
    size_m=5_000,
    scale=10,
    crs=proj
)
print("Started task:", task.id)